# 설정 및 모듈 임포트

In [ ]:
# --- 1. 라이브러리 및 개인 모듈 임포트 ---
import torch
import cv2
import time
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output
from pop import Camera

# 우리가 만든 .py 파일들을 임포트
import model as model_loader
import car_control
import image_processor

# --- 전역 변수 및 상태 관리 ---
is_running = False  # 주행 상태 플래그
stop_thread = False # 스레드 종료용 플래그
running_lock = threading.Lock() # 스레드 동시 접근 방지용

# --- AI 모델 설정 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 훈련 시 사용했던 설정값 (image_processor와 동일해야 함) ---
GRAYSCALE = False 
ROI_CROP_HEIGHT = 130
IMAGE_SIZE = (224, 224)

print("✅ 셀 1: 설정 및 모듈 임포트 완료. 다음 셀을 실행하세요.")

# AI 모델 로딩

In [ ]:
# --- 2. AI 모델 로딩 ---
MODEL_PATH = 'best_driving_model.pth'
model_loaded = False

try:
    # model.py에 정의된 DrivingModel 클래스로 모델 객체 생성
    driving_model = model_loader.DrivingModel(grayscale=GRAYSCALE).to(device)
    # 훈련된 가중치 불러오기
    driving_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    # 평가 모드로 전환 (매우 중요!)
    driving_model.eval()
    model_loaded = True
    print(f"✅ 셀 2: '{MODEL_PATH}' 모델 로딩 완료. 다음 셀을 실행하세요.")
except FileNotFoundError:
    print(f"❌ 오류: '{MODEL_PATH}' 파일을 찾을 수 없습니다. 훈련된 모델 파일이 있는지 확인하세요.")
except Exception as e:
    print(f"❌ 오류: 모델 로딩 중 문제 발생 - {e}")

# 사용자 인터페이스(UI) 위젯 생성 및 표시

In [ ]:
# --- 3. ipywidgets를 이용한 UI 생성 ---

# 위젯 정의
video_output = widgets.Output(layout={'width': '320px', 'height': '320px', 'border': '1px solid black'})
speed_slider = widgets.IntSlider(value=30, min=0, max=50, step=1, description='속도(Speed):', continuous_update=False)
start_button = widgets.Button(description="▶️ 주행 시작", button_style='success')
stop_button = widgets.Button(description="⏹️ 주행 정지", button_style='danger')
status_label = widgets.Label(value="대기 중. [▶️ 주행 시작] 버튼을 누르세요.")

# 버튼 클릭 시 실행될 함수 정의
def on_start_clicked(b):
    global is_running
    with running_lock:
        is_running = True
    car_control.set_speed(speed_slider.value) # 현재 슬라이더 값으로 속도 설정
    status_label.value = "주행 중... AI가 조향을 제어합니다."

def on_stop_clicked(b):
    global is_running
    with running_lock:
        is_running = False
    car_control.stop() # 자동차 정지
    status_label.value = "주행 정지. 다시 시작하려면 [▶️ 주행 시작]을 누르세요."
    
# 속도 슬라이더 값이 바뀔 때 실행될 함수 정의
def on_speed_changed(change):
    # 주행 중일 때만 속도를 변경
    if is_running:
        car_control.set_speed(change.new)

# 각 위젯에 함수 연결
start_button.on_click(on_start_clicked)
stop_button.on_click(on_stop_clicked)
speed_slider.observe(on_speed_changed, names='value')

# 위젯들을 화면에 표시
controls = widgets.HBox([start_button, stop_button])
ui_layout = widgets.VBox([video_output, status_label, speed_slider, controls])

print("✅ 셀 3: UI 생성 완료. 마지막 셀을 실행하여 자율주행을 시작하세요.")
display(ui_layout)

# 메인 자율주행 로직 실행 (백그라운드 스레드)

In [ ]:
# --- 4. 메인 자율주행 로직 (카메라 출력 수정본) ---

def driving_loop():
    """
    백그라운드 스레드에서 계속 실행될 메인 함수.
    pop.Util.imshow()를 사용하여 영상을 실시간으로 출력합니다.
    """
    global is_running, stop_thread
    
    try:
        # [수정] pop 라이브러리의 imshow 활성화 (사용자님 코드 참조)
        from pop import Util
        Util.enable_imshow()
        
        # 카메라 초기화
        cam = Camera(width=300, height=300) 
        
    except Exception as e:
        # video_output 위젯이 셀 3에서 정의되었으므로 직접 텍스트 출력
        print(f"카메라 또는 pop 라이브러리 초기화 실패: {e}")
        return

    while not stop_thread:
        # 현재 주행 상태 확인
        with running_lock:
            running = is_running

        frame = cam.value # 카메라에서 프레임 읽기
        
        # 시각화용 복사본 이미지 생성
        vis_frame = frame.copy()

        if running and model_loaded:
            # 1. 이미지 전처리
            input_tensor = image_processor.preprocess_for_model(frame).to(device)
            
            # 2. AI 모델 추론
            with torch.no_grad():
                steer_pred, _ = driving_model(input_tensor)
            
            # 3. 자동차 제어
            steering_value = steer_pred.item()
            car_control.set_steering(steering_value)
            
            # 4. 예측값 시각화
            pred_x_pos = int((steering_value + 1.0) / 2.0 * 300)
            y_pos = int(300 * 4/5)
            # OpenCV 함수를 이용해 복사본 이미지에 빨간 점 그리기
            cv2.circle(vis_frame, (pred_x_pos, y_pos), 7, (0,0,255), -1)

        # --- [핵심 수정] Jupyter 출력 방식 변경 ---
        # 이전의 복잡한 with video_output... 부분을 아래 한 줄로 대체합니다.
        # pop 라이브러리가 제공하는 전용 출력 함수를 사용합니다.
        Util.imshow("Camera Feed", vis_frame)
        # -----------------------------------------
            
        # OpenCV 창의 이벤트 처리 및 키 입력을 위해 짧은 대기시간 유지
        key = cv2.waitKey(1)
        # (선택사항) 만약의 경우를 대비한 비상 종료 키
        if key == ord('q'):
            with running_lock:
                is_running = False
            stop_thread = True # 스레드 종료 플래그 설정
            car_control.stop()

    # 루프 종료 시 정리
    cam.release()
    cv2.destroyAllWindows() # Util.imshow가 내부적으로 사용하는 창을 닫기 위함
    for i in range(5): cv2.waitKey(1)


# --- 스레드 시작 ---
# 이 셀을 재실행할 때 스레드가 중복 생성되는 것을 방지
if 'drive_thread' in locals() and drive_thread.is_alive():
    print("스레드가 이미 실행 중입니다. 중복 실행을 방지했습니다.")
    # 만약 재시작하고 싶다면, 먼저 커널을 재시작하세요.
else:
    print("🚀 자율주행 스레드를 시작합니다. UI를 사용하여 자동차를 제어하세요.")
    stop_thread = False
    drive_thread = threading.Thread(target=driving_loop)
    drive_thread.start()

# 이제 교차로를 정의할 시간. 

In [ ]:
# --- 4. 메인 자율주행 로직 (카메라 출력 수정본) ---

def driving_loop():
    """
    백그라운드 스레드에서 계속 실행될 메인 함수.
    pop.Util.imshow()를 사용하여 영상을 실시간으로 출력합니다.
    """
    global is_running, stop_thread
    
    try:
        # [수정] pop 라이브러리의 imshow 활성화 (사용자님 코드 참조)
        from pop import Util
        Util.enable_imshow()
        
        # 카메라 초기화
        cam = Camera(width=300, height=300) 
        
    except Exception as e:
        # video_output 위젯이 셀 3에서 정의되었으므로 직접 텍스트 출력
        print(f"카메라 또는 pop 라이브러리 초기화 실패: {e}")
        return

    while not stop_thread:
        # 현재 주행 상태 확인
        with running_lock:
            running = is_running

        frame = cam.value # 카메라에서 프레임 읽기
        
        # 시각화용 복사본 이미지 생성
        vis_frame = frame.copy()

        if running and model_loaded:
            input_tensor = image_processor.preprocess_for_model(frame).to(device)
            with torch.no_grad():
                # [수정] 이제 두 개의 출력을 모두 받습니다.
                steer_pred, inter_pred_logits = driving_model(input_tensor)

            steering_value = steer_pred.item()
            car_control.set_steering(steering_value)

            # [추가] 교차로 예측값을 처리하고 시각화합니다.
            predicted_intersection = torch.max(inter_pred_logits.data, 1)[1].item()

            # 예측값 시각화
            pred_x_pos = int((steering_value + 1.0) / 2.0 * 300)
            y_pos = int(300 * 4/5)
            cv2.circle(vis_frame, (pred_x_pos, y_pos), 7, (0,0,255), -1)

            # [추가] 교차로 예측 결과를 화면에 텍스트로 표시
            intersection_text = f"Intersection Prediction: {predicted_intersection}"
            cv2.putText(vis_frame, intersection_text, (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            

        # --- [핵심 수정] Jupyter 출력 방식 변경 ---
        # 이전의 복잡한 with video_output... 부분을 아래 한 줄로 대체합니다.
        # pop 라이브러리가 제공하는 전용 출력 함수를 사용합니다.
        Util.imshow("Camera Feed", vis_frame)
        # -----------------------------------------
            
        # OpenCV 창의 이벤트 처리 및 키 입력을 위해 짧은 대기시간 유지
        key = cv2.waitKey(1)
        # (선택사항) 만약의 경우를 대비한 비상 종료 키
        if key == ord('q'):
            with running_lock:
                is_running = False
            stop_thread = True # 스레드 종료 플래그 설정
            car_control.stop()

    # 루프 종료 시 정리
    cam.release()
    cv2.destroyAllWindows() # Util.imshow가 내부적으로 사용하는 창을 닫기 위함
    for i in range(5): cv2.waitKey(1)


# --- 스레드 시작 ---
# 이 셀을 재실행할 때 스레드가 중복 생성되는 것을 방지
if 'drive_thread' in locals() and drive_thread.is_alive():
    print("스레드가 이미 실행 중입니다. 중복 실행을 방지했습니다.")
    # 만약 재시작하고 싶다면, 먼저 커널을 재시작하세요.
else:
    print("🚀 자율주행 스레드를 시작합니다. UI를 사용하여 자동차를 제어하세요.")
    stop_thread = False
    drive_thread = threading.Thread(target=driving_loop)
    drive_thread.start()

In [ ]:
# 완벽한 하나의 샷으로 

In [ ]:
#################################################################
#                   자율주행 제어 프로그램 (최종본 v1.2)                  #
#################################################################

# --- 1. 라이브러리 및 개인 모듈 임포트 ---
print("1. 라이브러리 및 모듈을 불러오는 중...")
import torch, torch.nn as nn, os, cv2, numpy as np, shutil, time, threading
import ipywidgets as widgets
from IPython.display import display, clear_output
from pop import Camera, Util
# [추가] 슬라이딩 윈도우를 위한 deque 임포트
from collections import deque 

import model as model_loader, car_control, image_processor, maneuvers

# --- 2. 설정 및 전역 변수 ---
print("2. 프로그램 설정 및 변수를 초기화하는 중...")
# 상태 관리
car_state = 'IDLE'
drive_mode = 'loop'
stop_thread = False
state_lock = threading.Lock()
user_command = None

# AI 및 이미지 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRAYSCALE = False 
ROI_CROP_HEIGHT = 130
IMAGE_SIZE = (224, 224)
MODEL_PATH = 'best_driving_model.pth'

# [수정] 교차로 판단을 위한 슬라이딩 윈도우 설정
WINDOW_SIZE = 20  # 최근 몇 개의 프레임을 기억할지
DETECTION_THRESHOLD_PERCENT = 20 # 윈도우 안에서 몇 % 이상이 교차로여야 인정할지
recent_predictions = deque(maxlen=WINDOW_SIZE) # 예측 기록을 저장할 윈도우

# --- 3. 모델 로딩 ---
print(f"3. AI 모델 '{MODEL_PATH}'를 불러오는 중...")
model_loaded = False
try:
    driving_model = model_loader.DrivingModel(grayscale=GRAYSCALE).to(device)
    driving_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    driving_model.eval()
    model_loaded = True
    print(f"   - 모델 로딩 완료. 추론 디바이스: {device}")
except Exception as e:
    print(f"   - ❌ 모델 로딩 실패: {e}")

# --- 4. UI 위젯 및 콜백 함수 (이전과 동일) ---
print("4. 사용자 인터페이스(UI)를 생성하는 중...")
video_output = widgets.Output(layout={'width': '320px', 'height': '320px', 'border': '1px solid black'})
status_label = widgets.Label(value="대기 중. 모드와 속도를 설정하고 [시동 켜기]를 누르세요.")
speed_slider = widgets.IntSlider(value=30, min=0, max=50, step=1, description='속도:')
mode_selector = widgets.ToggleButtons(options=['외곽 순환', '교차로 주행'], description='주행 모드:')
start_button = widgets.Button(description="🚀 시동 켜기", button_style='success')
stop_button = widgets.Button(description="⏹️ 일시 정지", button_style='warning')
left_button = widgets.Button(description="← 좌회전", button_style='info', disabled=True)
straight_button = widgets.Button(description="↑ 직진", button_style='info', disabled=True)
right_button = widgets.Button(description="→ 우회전", button_style='info', disabled=True)

def set_drive_buttons_disabled(disabled):
    left_button.disabled = disabled; straight_button.disabled = disabled; right_button.disabled = disabled
def on_start_click(b):
    global car_state
    with state_lock:
        if car_state in ['IDLE', 'PAUSED']: car_state = 'LANE_FOLLOWING'; status_label.value = "주행 시작..."; car_control.set_speed(speed_slider.value)
def on_stop_click(b):
    global car_state
    with state_lock:
        if car_state != 'IDLE': car_state = 'PAUSED'; status_label.value = "일시 정지."; car_control.stop()
def on_maneuver_command(command):
    global car_state, user_command
    with state_lock:
        if car_state == 'WAITING_FOR_COMMAND':
            user_command = command; car_state = 'EXECUTING_MANEUVER'; set_drive_buttons_disabled(True); status_label.value = f"{command} 기동 실행 중..."
def on_mode_change(change):
    global drive_mode
    drive_mode = 'intersection' if change.new == '교차로 주행' else 'loop'; print(f"주행 모드가 '{drive_mode}'로 변경되었습니다.")

# --- 5. 메인 자율주행 로직을 담은 스레드 함수 ---
def driving_loop():
    global car_state, user_command, stop_thread, recent_predictions
    try:
        Util.enable_imshow(); cam = Camera(width=300, height=300)
    except Exception as e:
        print(f"카메라 초기화 실패: {e}"); return

    while not stop_thread:
        with state_lock: current_state_snapshot = car_state
        frame = cam.value; vis_frame = frame.copy()
        
        if current_state_snapshot == 'LANE_FOLLOWING':
            input_tensor = image_processor.preprocess_for_model(frame).to(device)
            with torch.no_grad():
                steer_pred, inter_pred = driving_model(input_tensor)
            
            steering_value = steer_pred.item()
            predicted_intersection = torch.max(inter_pred.data, 1)[1].item()
            car_control.set_steering(steering_value)
            
            # --- [핵심 수정] 교차로 판단 로직: 슬라이딩 윈도우 적용 ---
            detection_ratio = 0
            if drive_mode == 'intersection':
                # 1. 현재 예측을 윈도우에 추가
                recent_predictions.append(predicted_intersection)
                
                # 2. 윈도우가 가득 찼을 때만 판단 시작
                if len(recent_predictions) == WINDOW_SIZE:
                    # 3. 윈도우 내에서 교차로(0보다 큰 값)의 개수를 셈
                    intersection_count = sum(1 for pred in recent_predictions if pred > 0)
                    # 4. 비율 계산
                    detection_ratio = (intersection_count / WINDOW_SIZE) * 100
                    
                    # 5. 비율이 임계값을 넘으면 정지
                    if detection_ratio >= DETECTION_THRESHOLD_PERCENT:
                        with state_lock: car_state = 'WAITING_FOR_COMMAND'
                        car_control.stop()
                        status_label.value = f"교차로({predicted_intersection}) 확실! 방향 선택."
                        set_drive_buttons_disabled(False)
            # -----------------------------------------------------------
            
            pred_x_pos = int((steering_value + 1.0) / 2.0 * 300)
            cv2.circle(vis_frame, (pred_x_pos, int(300*4/5)), 7, (0,0,255), -1)
            cv2.putText(vis_frame, f"Inter. Ratio: {detection_ratio:.1f}%", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
        
        elif current_state_snapshot == 'WAITING_FOR_COMMAND' or current_state_snapshot == 'EXECUTING_MANEUVER':
            recent_predictions.clear() # 교차로에 진입하면 윈도우를 비워서 중복 판단 방지
            if user_command:
                if user_command == 'left': maneuvers.execute_left_turn()
                elif user_command == 'straight': maneuvers.execute_go_straight()
                elif user_command == 'right': maneuvers.execute_right_turn()
                with state_lock:
                    user_command = None; car_state = 'LANE_FOLLOWING'; status_label.value = "기동 완료. AI 자율주행 복귀."
                    car_control.set_speed(speed_slider.value)
        
        Util.imshow("Autonomous Driving v-Final", vis_frame)
        if cv2.waitKey(1) == 27: stop_thread = True
    
    cam.release(); car_control.stop(); cv2.destroyAllWindows();
    for i in range(5): cv2.waitKey(1)

# --- 6. 최종 UI 구성 및 스레드 시작 ---
print("5. UI 위젯에 기능(콜백 함수)을 연결하는 중...")
start_button.on_click(on_start_click)
stop_button.on_click(on_stop_click)
left_button.on_click(lambda b: on_maneuver_command('left'))
straight_button.on_click(lambda b: on_maneuver_command('straight'))
right_button.on_click(lambda b: on_maneuver_command('right'))
mode_selector.observe(on_mode_change, names='value')

top_controls = widgets.HBox([start_button, stop_button])
maneuver_controls = widgets.HBox([left_button, straight_button, right_button])
ui_layout = widgets.VBox([status_label, mode_selector, speed_slider, top_controls, maneuver_controls])

if model_loaded:
    print("6. 자율주행 백그라운드 스레드를 시작합니다...")
    stop_thread = False
    drive_thread = threading.Thread(target=driving_loop)
    drive_thread.start()
    display(video_output, ui_layout)
else:
    print("\n❌ 모델 로딩에 실패하여 프로그램을 시작할 수 없습니다.")

# v2

In [1]:
#################################################################
#                   자율주행 제어 프로그램 (최종본 v1.2)                  #
#################################################################

# --- 1. 라이브러리 및 개인 모듈 임포트 ---
print("1. 라이브러리 및 모듈을 불러오는 중...")
import torch, torch.nn as nn, os, cv2, numpy as np, shutil, time, threading
import ipywidgets as widgets
from IPython.display import display, clear_output
from pop import Camera, Util
# [추가] 슬라이딩 윈도우를 위한 deque 임포트
from collections import deque 

import model as model_loader, car_control, image_processor, maneuvers

# --- 2. 설정 및 전역 변수 ---
print("2. 프로그램 설정 및 변수를 초기화하는 중...")
# 상태 관리
MANEUVER_BLEND_START_PHASE = 2.0  # 몇 번째 단계(Phase)부터 AI와 타협을 시작할지 (0, 1.0, 2.0 이렇게 올라감. )
MANEUVER_BLEND_RATIO = 0.25      # 타협 비율 (0.0: 100% AI, 1.0: 100% 하드코딩)

car_state = 'IDLE'
drive_mode = 'loop'
stop_thread = False
state_lock = threading.Lock()
user_command = None

# AI 및 이미지 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GRAYSCALE = False 
ROI_CROP_HEIGHT = 130
IMAGE_SIZE = (224, 224)
MODEL_PATH = 'best_driving_model.pth'

# [수정] 교차로 판단을 위한 슬라이딩 윈도우 설정
WINDOW_SIZE = 15  # 최근 몇 개의 프레임을 기억할지
DETECTION_THRESHOLD_PERCENT = 20 # 윈도우 안에서 몇 % 이상이 교차로여야 인정할지
recent_predictions = deque(maxlen=WINDOW_SIZE) # 예측 기록을 저장할 윈도우
maneuver_recipe = None; maneuver_phase_index = 0; maneuver_phase_start_time = 0


# --- 3. 모델 로딩 ---
print(f"3. AI 모델 '{MODEL_PATH}'를 불러오는 중...")
model_loaded = False
try:
    driving_model = model_loader.DrivingModel(grayscale=GRAYSCALE).to(device)
    driving_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    driving_model.eval()
    model_loaded = True
    print(f"   - 모델 로딩 완료. 추론 디바이스: {device}")
except Exception as e:
    print(f"   - ❌ 모델 로딩 실패: {e}")

# --- 4. UI 위젯 및 콜백 함수 (이전과 동일) ---
print("4. 사용자 인터페이스(UI)를 생성하는 중...")
video_output = widgets.Output(layout={'width': '320px', 'height': '320px', 'border': '1px solid black'})
status_label = widgets.Label(value="대기 중. 모드와 속도를 설정하고 [시동 켜기]를 누르세요.")
speed_slider = widgets.IntSlider(value=30, min=0, max=50, step=1, description='속도:')
mode_selector = widgets.ToggleButtons(options=['외곽 순환', '교차로 주행'], description='주행 모드:')
start_button = widgets.Button(description="🚀 시동 켜기", button_style='success')
stop_button = widgets.Button(description="⏹️ 일시 정지", button_style='warning')
left_button = widgets.Button(description="← 좌회전", button_style='info', disabled=True)
straight_button = widgets.Button(description="↑ 직진", button_style='info', disabled=True)
right_button = widgets.Button(description="→ 우회전", button_style='info', disabled=True)

def set_drive_buttons_disabled(disabled):
    left_button.disabled = disabled; straight_button.disabled = disabled; right_button.disabled = disabled
def on_start_click(b):
    global car_state
    with state_lock:
        if car_state in ['IDLE', 'PAUSED']: car_state = 'LANE_FOLLOWING'; status_label.value = "주행 시작..."; car_control.set_speed(speed_slider.value)
def on_stop_click(b):
    global car_state
    with state_lock:
        if car_state != 'IDLE': car_state = 'PAUSED'; status_label.value = "일시 정지."; car_control.stop()
def on_maneuver_command(command):
    global car_state, user_command
    with state_lock:
        if car_state == 'WAITING_FOR_COMMAND':
            user_command = command; car_state = 'EXECUTING_MANEUVER'; set_drive_buttons_disabled(True); status_label.value = f"{command} 기동 실행 중..."
def on_mode_change(change):
    global drive_mode
    drive_mode = 'intersection' if change.new == '교차로 주행' else 'loop'; print(f"주행 모드가 '{drive_mode}'로 변경되었습니다.")

# --- 5. 메인 자율주행 로직을 담은 스레드 함수 ---
# 이 함수는 main.ipynb의 셀 6에서 스레드로 실행될 부분입니다.
# 상단에 정의된 모든 전역 변수(car_state, lock, device 등)를 사용합니다.

def driving_loop():
    """
    백그라운드 스레드에서 계속 실행될 메인 함수.
    카메라 영상 획득, AI 추론, 자동차 제어, 영상 출력을 담당.
    """
    # 이 함수 안에서 사용할 전역 변수들을 명시
    global car_state, user_command, stop_thread, recent_predictions
    global maneuver_recipe, maneuver_phase_index, maneuver_phase_start_time
    
    # --- 초기화 ---
    try:
        Util.enable_imshow()
        cam = Camera(width=300, height=300)
        print("-> 카메라가 성공적으로 초기화되었습니다.")
    except Exception as e:
        print(f"❌ 카메라 초기화 실패: {e}")
        return

    # --- 메인 루프 시작 ---
    while not stop_thread:
        # 현재 자동차의 상태를 안전하게 읽어옴
        with state_lock:
            current_state_snapshot = car_state
        
        # 1. 카메라에서 프레임 읽기 및 시각화용 복사본 생성
        frame = cam.value
        vis_frame = frame.copy()
        
        # 2. AI는 항상 최신 상황을 예측 (어떤 상태이든)
        #    '타협' 단계에서 AI의 실시간 예측값이 필요하기 때문
        input_tensor = image_processor.preprocess_for_model(frame).to(device)
        with torch.no_grad():
            steer_pred, inter_pred = driving_model(input_tensor)
        
        ai_steering = steer_pred.item()
        predicted_intersection = torch.max(inter_pred.data, 1)[1].item()

        # 3. 자동차의 현재 '상태'에 따라 다른 행동 수행
        
        # --- 상태: LANE_FOLLOWING (AI 자율주행) ---
        if current_state_snapshot == 'LANE_FOLLOWING':
            # AI의 예측값으로 조향 및 속도 제어
            car_control.set_steering(ai_steering)
            
            # 교차로 주행 모드일 때만 교차로 감지 로직 활성화
            detection_ratio = 0
            if drive_mode == 'intersection':
                # 슬라이딩 윈도우에 현재 예측 추가
                recent_predictions.append(predicted_intersection)
                
                if len(recent_predictions) == WINDOW_SIZE:
                    intersection_count = sum(1 for pred in recent_predictions if pred > 0)
                    detection_ratio = (intersection_count / WINDOW_SIZE) * 100
                    
                    # 비율이 임계값을 넘으면 상태 변경 및 정지
                    if detection_ratio >= DETECTION_THRESHOLD_PERCENT:
                        with state_lock:
                            car_state = 'WAITING_FOR_COMMAND'
                        car_control.stop()
                        status_label.value = f"교차로({predicted_intersection}) 확실! 방향 선택."
                        set_drive_buttons_disabled(False)
            
            # 시각화: AI 예측 조향값(빨간점)과 교차로 감지 비율 표시
            pred_x_pos = int((ai_steering + 1.0) / 2.0 * 300)
            cv2.circle(vis_frame, (pred_x_pos, int(300*4/5)), 7, (0,0,255), -1)
            cv2.putText(vis_frame, f"Inter. Ratio: {detection_ratio:.1f}%", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
        
        # --- 상태: EXECUTING_MANEUVER (하이브리드 기동) ---
        elif current_state_snapshot == 'EXECUTING_MANEUVER':
            # 기동이 처음 시작되는 순간, 레시피를 가져옴
            if maneuver_recipe is None:
                if user_command == 'left': maneuver_recipe = maneuvers.get_left_turn_recipe()
                elif user_command == 'straight': maneuver_recipe = maneuvers.get_go_straight_recipe()
                elif user_command == 'right': maneuver_recipe = maneuvers.get_right_turn_recipe()
                maneuver_phase_index = 0
                maneuver_phase_start_time = time.time()

            # 현재 단계가 레시피를 벗어나면 기동 종료
            if maneuver_phase_index >= len(maneuver_recipe):
                with state_lock:
                    user_command = None
                    maneuver_recipe = None
                    car_state = 'LANE_FOLLOWING'
                status_label.value = "기동 완료. AI 자율주행 복귀."
                car_control.set_speed(speed_slider.value)
                continue

            target_steer, target_speed, phase_duration = maneuver_recipe[maneuver_phase_index]
            
            # 현재 단계 시간이 다 지났으면 다음 단계로
            if time.time() - maneuver_phase_start_time > phase_duration:
                maneuver_phase_index += 1
                maneuver_phase_start_time = time.time()
                continue
            
            # 조향값 '타협' 로직
            alpha = 1.0 # 기본값은 100% 하드코딩
            if maneuver_phase_index >= MANEUVER_BLEND_START_PHASE:
                alpha = MANEUVER_BLEND_RATIO # 타협 단계에서는 설정된 비율 사용
            
            final_steering = (alpha * target_steer) + ((1 - alpha) * ai_steering)
            
            car_control.set_steering(final_steering)
            car_control.set_speed(target_speed)

            # 시각화: 현재 기동 상태와 최종 조향값(초록점) 표시
            cv2.putText(vis_frame, f"Maneuver: Phase {maneuver_phase_index}, Blend: {alpha*100:.0f}%", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
            final_x_pos = int((final_steering + 1.0) / 2.0 * 300)
            cv2.circle(vis_frame, (final_x_pos, int(300*4/5)), 7, (0,255,0), -1)

        # --- 기타 상태 ---
        elif current_state_snapshot == 'WAITING_FOR_COMMAND':
            recent_predictions.clear() # 윈도우 비워서 중복 감지 방지
            # UI 콜백이 상태를 바꿀 때까지 대기
        
        elif current_state_snapshot in ['IDLE', 'PAUSED']:
            recent_predictions.clear() # 윈도우 비우기
            # 제어는 이미 멈춰있으므로 대기

        # 4. 최종 가공된 이미지를 화면에 출력
        Util.imshow("Autonomous Driving", vis_frame)
        
        # 5. 비상 종료(ESC) 키 확인
        if cv2.waitKey(1) == 27:
            stop_thread = True
    
    # --- 루프 종료 후 정리 ---
    print("Driving loop is shutting down...")
    cam.release()
    car_control.stop()
    cv2.destroyAllWindows()
    for i in range(5): cv2.waitKey(1)
    

# --- 6. 최종 UI 구성 및 스레드 시작 ---
print("5. UI 위젯에 기능(콜백 함수)을 연결하는 중...")
start_button.on_click(on_start_click)
stop_button.on_click(on_stop_click)
left_button.on_click(lambda b: on_maneuver_command('left'))
straight_button.on_click(lambda b: on_maneuver_command('straight'))
right_button.on_click(lambda b: on_maneuver_command('right'))
mode_selector.observe(on_mode_change, names='value')

top_controls = widgets.HBox([start_button, stop_button])
maneuver_controls = widgets.HBox([left_button, straight_button, right_button])
ui_layout = widgets.VBox([status_label, mode_selector, speed_slider, top_controls, maneuver_controls])

if model_loaded:
    print("6. 자율주행 백그라운드 스레드를 시작합니다...")
    stop_thread = False
    drive_thread = threading.Thread(target=driving_loop)
    drive_thread.start()
    display(video_output, ui_layout)
else:
    print("\n❌ 모델 로딩에 실패하여 프로그램을 시작할 수 없습니다.")

1. 라이브러리 및 모듈을 불러오는 중...
✅ AutoCar 객체가 성공적으로 초기화되었습니다.
2. 프로그램 설정 및 변수를 초기화하는 중...
3. AI 모델 'best_driving_model.pth'를 불러오는 중...
   - 모델 로딩 완료. 추론 디바이스: cuda
4. 사용자 인터페이스(UI)를 생성하는 중...
5. UI 위젯에 기능(콜백 함수)을 연결하는 중...
6. 자율주행 백그라운드 스레드를 시작합니다...


Output(layout=Layout(border='1px solid black', height='320px', width='320px'))

-> 카메라가 성공적으로 초기화되었습니다.


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…